# senti: score the whole catalogue with audio

Puts the **audio** into the live scores. For every song on the site that has an Apple preview, it listens to the
30-second clip, predicts valence and arousal from the audio with the head chosen in `colab_audio_fusion`, and blends
that with the song's lyrics score using the fusion weights learned there.

**Before running**
1. Run `colab_audio_fusion_fixed.ipynb` first (it now also saves `colab_results/audio_head.joblib`).
2. Upload `catalog_previews.csv` (from Claude: titles, artists, lyrics scores and preview links, no lyrics) to
   your Drive folder `moodify`.
3. Runtime › Change runtime type › **T4 GPU**, then Runtime › Run all.

Audio is streamed into memory and thrown away after each song; only the features (numbers) are cached in
`MyDrive/moodify/audio_cache/`, so a re-run continues where it stopped. The output,
`colab_results/catalog_audio_scores.csv`, holds titles, artists and numbers only. Send it to Claude.

In [ ]:
# Settings
DRIVE_DIR = "/content/drive/MyDrive/moodify"
INPUT = "catalog_previews.csv"
MERT_SLICE, MERT_HOP = 5.0, 2.5     # seconds, as in the fusion notebook
CLAP_SLICE, CLAP_HOP = 10.0, 5.0

In [ ]:
# MERT ships its own model code, written for 2024-era transformers; pin a version from then.
# 4.45.2: the newest MERT works with that installs on Colab's Python 3.13 (4.44 has no build for it).
!pip -q install "transformers==4.45.2" nnAudio librosa
from google.colab import drive
drive.mount("/content/drive")
import os, io, json, time, tempfile, requests, joblib
import numpy as np, pandas as pd, torch, librosa
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUT_DIR = os.path.join(DRIVE_DIR, "colab_results"); CACHE = os.path.join(DRIVE_DIR, "audio_cache")
os.makedirs(CACHE, exist_ok=True)
head = joblib.load(os.path.join(OUT_DIR, "audio_head.joblib"))
BEST, weights = head["features"], head["fusion_weights"]
cat = pd.read_csv(os.path.join(DRIVE_DIR, INPUT))
cat["key"] = cat["title"] + "|" + cat["artist"]
print(f"{len(cat)} songs with previews; audio features: {BEST}; device: {dev}")
print("fusion weights:", json.dumps(weights))

## 1. Audio features for every preview (cached in Drive)

In [ ]:
# Fast path: previews are downloaded and decoded ~32 at a time in parallel (that, not the GPU, is the slow part),
# and the GPU sees large half-precision batches of slices from many songs at once.
import subprocess
from concurrent.futures import ThreadPoolExecutor
WORKERS, SONGS_PER_BATCH = 32, 48

def slices(y, sr, length, hop):
    n, h = int(length * sr), int(hop * sr)
    if len(y) <= n: return [np.pad(y, (0, n - len(y)))]
    return [y[s:s + n] for s in range(0, len(y) - n + 1, h)]

amp = torch.autocast("cuda", dtype=torch.float16) if dev.type == "cuda" else torch.autocast("cpu", enabled=False)

if BEST.startswith("MERT"):
    from transformers import AutoModel, Wav2Vec2FeatureExtractor
    L = int(BEST.split()[2])
    proc = Wav2Vec2FeatureExtractor.from_pretrained("m-a-p/MERT-v1-95M", trust_remote_code=True)
    model = AutoModel.from_pretrained("m-a-p/MERT-v1-95M", trust_remote_code=True).to(dev).eval()
    SR, LEN, HOP, TAG = 24000, MERT_SLICE, MERT_HOP, "catalog_mert95m"
    def embed_slices(sl):                       # list of equal-length slices -> [n, layers, 768]
        x = proc(sl, sampling_rate=SR, return_tensors="pt", padding=True)
        out = []
        with torch.no_grad(), amp:
            for k in range(0, len(sl), 64):
                hs = model(x["input_values"][k:k + 64].to(dev), output_hidden_states=True).hidden_states
                out.append(torch.stack([h.mean(1) for h in hs], 1).float().cpu())
        return torch.cat(out).numpy()
    pick = (lambda e: e[0, L]) if BEST.endswith(" mean") else (lambda e: e[:, L].reshape(-1))
else:
    from transformers import ClapModel, ClapProcessor
    proc = ClapProcessor.from_pretrained("laion/larger_clap_music")
    model = ClapModel.from_pretrained("laion/larger_clap_music").to(dev).eval()
    SR, LEN, HOP, TAG = 48000, CLAP_SLICE, CLAP_HOP, "catalog_clap_music"
    def embed_slices(sl):                       # -> [n, 512]
        out = []
        for k in range(0, len(sl), 128):
            part = sl[k:k + 128]
            try:
                x = proc(audio=part, sampling_rate=SR, return_tensors="pt")
            except (TypeError, ValueError):
                x = proc(audios=part, sampling_rate=SR, return_tensors="pt")
            with torch.no_grad(), amp:
                o = model.get_audio_features(**{k2: v.to(dev) for k2, v in x.items()})
            out.append((o if torch.is_tensor(o) else o.pooler_output).float().cpu())
        return torch.cat(out).numpy()
    pick = (lambda e: e[0]) if BEST == "CLAP mean" else (lambda e: e.reshape(-1))

def fetch(url):
    """Download one preview and decode it to mono float32 at SR with ffmpeg (runs in parallel threads)."""
    r = requests.get(url, timeout=30); r.raise_for_status()
    pcm = subprocess.run(["ffmpeg", "-loglevel", "error", "-i", "pipe:0", "-f", "f32le", "-ac", "1", "-ar", str(SR), "pipe:1"],
                         input=r.content, capture_output=True, check=True).stdout
    return np.frombuffer(pcm, dtype=np.float32)

path = os.path.join(CACHE, f"{TAG}.npz")
done = dict(np.load(path, allow_pickle=True)) if os.path.exists(path) else {}
todo = cat[~cat["key"].isin(done.keys())][["key", "preview_url"]].to_numpy()
print(f"{len(done)} cached, {len(todo)} to go")
t0, skipped = time.time(), 0
with ThreadPoolExecutor(WORKERS) as pool:
    for b in range(0, len(todo), SONGS_PER_BATCH):
        batch = todo[b:b + SONGS_PER_BATCH]
        futs = [pool.submit(fetch, u) for _, u in batch]
        keys, sl, owner = [], [], []
        for (key, _), f in zip(batch, futs):
            try:
                y = f.result()
            except Exception as e:
                skipped += 1; print("skip", key, repr(e)[:100]); continue
            s = slices(y, SR, LEN, HOP); keys.append(key); sl += s; owner += [len(keys) - 1] * len(s)
        if not keys: continue
        per = embed_slices(sl); owner = np.array(owner)
        for j, key in enumerate(keys):
            e = per[owner == j]
            done[key] = np.stack([e.mean(0), e.std(0)]).astype(np.float16)
        if (b // SONGS_PER_BATCH) % 10 == 9 or b + SONGS_PER_BATCH >= len(todo):
            np.savez(path, **done)
            n = min(b + SONGS_PER_BATCH, len(todo)); rate = n / max(time.time() - t0, 1e-6)
            print(f"  {n}/{len(todo)}  ({(time.time() - t0) / 60:.1f} min, {rate:.1f} songs/s, {skipped} skipped)")
np.savez(path, **done)

## 2. Audio scores, blended with the lyrics scores

In [ ]:
have = cat[cat["key"].isin(done.keys())].copy()
X = np.stack([pick(done[k].astype(np.float32)) for k in have["key"]])
audio = np.clip(head["ridge"].predict(head["scaler"].transform(X)), -1, 1)
have["audio_v"], have["audio_a"] = audio[:, 0], audio[:, 1]
for ax, lyr, aud in [("valence", "lyrics_v", "audio_v"), ("arousal", "lyrics_a", "audio_a")]:
    w = weights[ax]
    have[f"fused_{ax[0]}"] = np.clip(w["lyrics"] * have[lyr] + w["audio"] * have[aud] + w["bias"], -1, 1)
out = have[["title", "artist", "lyrics_v", "lyrics_a", "audio_v", "audio_a", "fused_v", "fused_a"]].round(4)
out.to_csv(os.path.join(OUT_DIR, "catalog_audio_scores.csv"), index=False)
print(f"scored {len(out)} of {len(cat)} songs -> colab_results/catalog_audio_scores.csv")
moved = np.hypot(out["fused_v"] - out["lyrics_v"], out["fused_a"] - out["lyrics_a"])
print(f"median move on the map: {moved.median():.3f} (map runs -1..1); biggest movers:")
display(out.assign(moved=moved).sort_values("moved", ascending=False).head(15))